# 07. Final Comparison and Error Analysis

So sánh baseline với PhoBERT, sau đó xem các hướng lỗi chính.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"
REPORTS_DIR = ROOT / "outputs" / "reports"
METRICS_DIR = ROOT / "outputs" / "metrics"
PREDICTIONS_DIR = ROOT / "outputs" / "predictions"

## 2. Kiểm tra output Stage 7 bắt buộc

Các file này phải tồn tại sau khi chạy:

```powershell
python scripts/run_final_comparison.py
```

In [ ]:
required_files = {
    "final_model_comparison": TABLES_DIR / "final_model_comparison.csv",
    "final_robustness_comparison": TABLES_DIR / "final_robustness_comparison.csv",
    "final_per_class_comparison": TABLES_DIR / "final_per_class_comparison.csv",
    "error_examples_sentiment": TABLES_DIR / "error_examples_sentiment.csv",
    "error_examples_topic": TABLES_DIR / "error_examples_topic.csv",
    "error_transition_summary_sentiment": TABLES_DIR / "error_transition_summary_sentiment.csv",
    "error_transition_summary_topic": TABLES_DIR / "error_transition_summary_topic.csv",
    "final_clean_macro_f1_figure": FIGURES_DIR / "final_clean_macro_f1_comparison.png",
    "final_robustness_macro_f1_sentiment": FIGURES_DIR / "final_robustness_macro_f1_sentiment.png",
    "final_robustness_macro_f1_topic": FIGURES_DIR / "final_robustness_macro_f1_topic.png",
    "final_robustness_drop_sentiment": FIGURES_DIR / "final_robustness_drop_sentiment.png",
    "final_robustness_drop_topic": FIGURES_DIR / "final_robustness_drop_topic.png",
    "final_report": REPORTS_DIR / "final_comparison_report.md",
    "final_metrics_json": METRICS_DIR / "final_comparison_metrics.json",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing Stage 7 files: {missing}")

print("All required Stage 7 files exist.")

## 3. Load dữ liệu tổng hợp

In [ ]:
final_model_comparison = pd.read_csv(TABLES_DIR / "final_model_comparison.csv")
final_robustness_comparison = pd.read_csv(TABLES_DIR / "final_robustness_comparison.csv")
final_per_class_comparison = pd.read_csv(TABLES_DIR / "final_per_class_comparison.csv")
error_examples_sentiment = pd.read_csv(TABLES_DIR / "error_examples_sentiment.csv")
error_examples_topic = pd.read_csv(TABLES_DIR / "error_examples_topic.csv")
transition_sentiment = pd.read_csv(TABLES_DIR / "error_transition_summary_sentiment.csv")
transition_topic = pd.read_csv(TABLES_DIR / "error_transition_summary_topic.csv")

display(Markdown("### final_model_comparison"))
display(final_model_comparison)

display(Markdown("### final_robustness_comparison preview"))
display(final_robustness_comparison.head(20))

display(Markdown("### final_per_class_comparison preview"))
display(final_per_class_comparison.head(20))

## 4. Clean test comparison

Nội dung: trả lời RQ1 — PhoBERT có vượt baseline trên dữ liệu sạch không?

In [ ]:
clean_comparison = final_model_comparison.copy()
clean_comparison = clean_comparison.sort_values(["task", "rank_macro_f1"])

display(clean_comparison)

best_clean = clean_comparison[clean_comparison["is_best_clean_macro_f1"] == True].copy()
display(Markdown("### Best model on clean test by task"))
display(best_clean[["task", "model", "accuracy", "macro_f1", "weighted_f1"]])

for _, row in best_clean.iterrows():
    display(Markdown(
        f"- **{row['task']}**: best clean model is **{row['model']}** "
        f"with Macro-F1 = **{row['macro_f1']:.4f}** and Accuracy = **{row['accuracy']:.4f}**."
    ))

### Clean Macro-F1 figure

In [ ]:
display(Image(filename=str(FIGURES_DIR / "final_clean_macro_f1_comparison.png")))

## 5. Robustness comparison by variant

Nội dung: so sánh toàn bộ model trên từng loại noise.

In [ ]:
variant_order = [
    "clean",
    "typo_light",
    "typo_medium",
    "teencode_light",
    "mixed_light",
    "no_accent",
    "mixed_no_accent",
]

robustness = final_robustness_comparison.copy()
robustness["variant"] = pd.Categorical(
    robustness["variant"],
    categories=variant_order,
    ordered=True,
)
robustness = robustness.sort_values(["task", "variant", "rank_macro_f1_within_variant"])

display(robustness)

## 6. Best model theo từng task và variant

Nội dung: trả lời RQ3 — PhoBERT có bền hơn baseline không?

In [ ]:
best_by_variant = robustness[robustness["is_best_model_for_variant"] == True].copy()
best_by_variant = best_by_variant.sort_values(["task", "variant"])

display(best_by_variant[[
    "task",
    "variant",
    "model",
    "model_family",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "macro_f1_drop",
    "macro_f1_relative_drop_percent",
]])

for task, task_df in best_by_variant.groupby("task"):
    display(Markdown(f"### Best model by variant — {task}"))
    display(task_df[["variant", "model", "macro_f1", "macro_f1_drop"]])

## 7. Macro-F1 theo variant

Nội dung: trực quan hóa mô hình nào giảm mạnh khi gặp noise.

In [ ]:
display(Markdown("### Sentiment"))
display(Image(filename=str(FIGURES_DIR / "final_robustness_macro_f1_sentiment.png")))

display(Markdown("### Topic"))
display(Image(filename=str(FIGURES_DIR / "final_robustness_macro_f1_topic.png")))

## 8. Macro-F1 drop theo variant

Nội dung: trả lời RQ2 — loại noise nào gây suy giảm mạnh nhất.

In [ ]:
display(Markdown("### Sentiment"))
display(Image(filename=str(FIGURES_DIR / "final_robustness_drop_sentiment.png")))

display(Markdown("### Topic"))
display(Image(filename=str(FIGURES_DIR / "final_robustness_drop_topic.png")))

## 9. Worst drop theo từng model

Nội dung: xác định mỗi model yếu nhất ở loại noise nào.

In [ ]:
worst_drop = (
    robustness[robustness["variant"] != "clean"]
    .sort_values(["task", "model", "macro_f1_drop"], ascending=[True, True, False])
    .groupby(["task", "model"])
    .head(1)
    .reset_index(drop=True)
)

display(worst_drop[[
    "task",
    "model",
    "variant",
    "clean_macro_f1",
    "variant_macro_f1",
    "macro_f1_drop",
    "macro_f1_relative_drop_percent",
]])

for _, row in worst_drop.iterrows():
    display(Markdown(
        f"- **{row['task']} / {row['model']}**: worst variant = `{row['variant']}`, "
        f"Macro-F1 drop = **{row['macro_f1_drop']:.4f}** "
        f"({row['macro_f1_relative_drop_percent']:.2f}%)."
    ))

## 10. PhoBERT vs best baseline trên no_accent

Nội dung: kiểm tra kết luận quan trọng nhất của robustness: PhoBERT có thua char-level SVM khi mất dấu không?

In [ ]:
focus_variants = ["no_accent", "mixed_no_accent"]

focus = robustness[robustness["variant"].isin(focus_variants)].copy()

display(focus[[
    "task",
    "variant",
    "model",
    "model_family",
    "accuracy",
    "macro_f1",
    "macro_f1_drop",
    "rank_macro_f1_within_variant",
]])

phobert_focus = focus[focus["model"] == "phobert"][
    ["task", "variant", "macro_f1"]
].rename(columns={"macro_f1": "phobert_macro_f1"})

best_baseline_focus = (
    focus[focus["model_family"] == "baseline"]
    .sort_values(["task", "variant", "macro_f1"], ascending=[True, True, False])
    .groupby(["task", "variant"])
    .head(1)
    [["task", "variant", "model", "macro_f1"]]
    .rename(columns={"model": "best_baseline_model", "macro_f1": "best_baseline_macro_f1"})
)

focus_compare = best_baseline_focus.merge(
    phobert_focus,
    on=["task", "variant"],
    how="inner",
)

focus_compare["phobert_minus_best_baseline"] = (
    focus_compare["phobert_macro_f1"] - focus_compare["best_baseline_macro_f1"]
)

display(Markdown("### PhoBERT vs best baseline on no-accent variants"))
display(focus_compare)

for _, row in focus_compare.iterrows():
    direction = "higher" if row["phobert_minus_best_baseline"] > 0 else "lower"
    display(Markdown(
        f"- **{row['task']} / {row['variant']}**: PhoBERT Macro-F1 is **{direction}** than "
        f"`{row['best_baseline_model']}` by **{row['phobert_minus_best_baseline']:.4f}**."
    ))

## 11. Per-class comparison

Nội dung: phân tích lớp nào yếu và lớp nào bị ảnh hưởng bởi noise.

In [ ]:
per_class = final_per_class_comparison.copy()
summary_labels = {"accuracy", "macro avg", "weighted avg"}
per_class_only = per_class[~per_class["label"].isin(summary_labels)].copy()

display(per_class_only.head(30))

for task in per_class_only["task"].dropna().unique():
    display(Markdown(f"### Clean per-class F1 — {task}"))
    clean_pc = per_class_only[
        (per_class_only["task"] == task) &
        (per_class_only["variant"] == "clean")
    ].copy()
    pivot = clean_pc.pivot_table(
        index="label",
        columns="model",
        values="f1_score",
        observed=False,
    )
    display(pivot)

## 12. Per-class F1 drop: clean → no_accent

Nội dung: xem lớp nào của PhoBERT sụp mạnh nhất khi bỏ dấu.

In [ ]:
phobert_pc = per_class_only[per_class_only["model"] == "phobert"].copy()

clean_pc = phobert_pc[phobert_pc["variant"] == "clean"][
    ["task", "label", "precision", "recall", "f1_score"]
].rename(
    columns={
        "precision": "clean_precision",
        "recall": "clean_recall",
        "f1_score": "clean_f1",
    }
)

no_accent_pc = phobert_pc[phobert_pc["variant"] == "no_accent"][
    ["task", "label", "precision", "recall", "f1_score"]
].rename(
    columns={
        "precision": "no_accent_precision",
        "recall": "no_accent_recall",
        "f1_score": "no_accent_f1",
    }
)

pc_drop = clean_pc.merge(no_accent_pc, on=["task", "label"], how="inner")
pc_drop["f1_drop"] = pc_drop["clean_f1"] - pc_drop["no_accent_f1"]
pc_drop["relative_f1_drop_percent"] = pc_drop.apply(
    lambda row: 0.0 if row["clean_f1"] == 0 else round(row["f1_drop"] / row["clean_f1"] * 100, 4),
    axis=1,
)

pc_drop = pc_drop.sort_values(["task", "f1_drop"], ascending=[True, False])

display(pc_drop)

for task, group_df in pc_drop.groupby("task"):
    worst = group_df.iloc[0]
    display(Markdown(
        f"- **{task}**: largest PhoBERT F1 drop under `no_accent` is `{worst['label']}` "
        f"with drop = **{worst['f1_drop']:.4f}**."
    ))

## 13. Error transition summary — Sentiment

Nội dung: trả lời RQ4 — mô hình nhầm từ lớp nào sang lớp nào?

In [ ]:
transition_sentiment_view = transition_sentiment.copy()
display(transition_sentiment_view.head(30))

for variant in transition_sentiment_view["variant"].dropna().unique():
    display(Markdown(f"### Sentiment transitions — {variant}"))
    display(
        transition_sentiment_view[transition_sentiment_view["variant"] == variant]
        .sort_values("count", ascending=False)
        .head(10)
    )

## 14. Error transition summary — Topic

In [ ]:
transition_topic_view = transition_topic.copy()
display(transition_topic_view.head(30))

for variant in transition_topic_view["variant"].dropna().unique():
    display(Markdown(f"### Topic transitions — {variant}"))
    display(
        transition_topic_view[transition_topic_view["variant"] == variant]
        .sort_values("count", ascending=False)
        .head(10)
    )

## 15. Error examples — Sentiment

Bảng này là mẫu định tính: **clean đúng nhưng noisy sai**.

Ghi chú:

```text
Các ví dụ này dùng để minh họa lỗi.
Không dùng bảng này để tính tỷ lệ lỗi vì mỗi variant chỉ lấy mẫu tối đa 30 dòng.
```

In [ ]:
display(error_examples_sentiment.head(30))

for variant in error_examples_sentiment["variant"].dropna().unique():
    display(Markdown(f"### Sentiment error examples — {variant}"))
    cols = [
        "original_text",
        "text",
        "y_true_name",
        "clean_y_pred_name",
        "y_pred_name",
        "error_type",
    ]
    cols = [c for c in cols if c in error_examples_sentiment.columns]
    display(error_examples_sentiment[error_examples_sentiment["variant"] == variant][cols].head(10))

## 16. Error examples — Topic

In [ ]:
display(error_examples_topic.head(30))

for variant in error_examples_topic["variant"].dropna().unique():
    display(Markdown(f"### Topic error examples — {variant}"))
    cols = [
        "original_text",
        "text",
        "y_true_name",
        "clean_y_pred_name",
        "y_pred_name",
        "error_type",
    ]
    cols = [c for c in cols if c in error_examples_topic.columns]
    display(error_examples_topic[error_examples_topic["variant"] == variant][cols].head(10))

## 17. Kết luận trả lời câu hỏi nghiên cứu

### RQ1 — PhoBERT có vượt baseline trên clean test không?

```text
Có. PhoBERT đạt Macro-F1 cao nhất trên clean test ở cả sentiment và topic.
```

### RQ2 — Loại noise nào gây suy giảm mạnh nhất?

```text
no_accent và mixed_no_accent là hai loại noise gây suy giảm mạnh nhất.
```

### RQ3 — PhoBERT có bền hơn baseline không?

```text
Cần chia theo điều kiện:
- Với clean và noise nhẹ, PhoBERT tốt nhất.
- Với no_accent và mixed_no_accent, TF-IDF char SVM bền hơn PhoBERT.
```

### RQ4 — Lớp nào và hướng nhầm nào nổi bật?

```text
Sentiment:
- Nhiều mẫu negative/positive bị kéo sang neutral khi mất dấu.

Topic:
- Nhiều mẫu lecturer/training_program bị kéo sang others khi mất dấu.
```

Diễn giải an toàn:

```text
Clean test đánh giá hiệu năng in-distribution.
Noisy test đánh giá robustness dưới nhiễu có kiểm soát.
Noisy test không được dùng để train hoặc chọn mô hình.
```

## 18. Kết luận Stage 7

Stage 7 hoàn thành khi có đủ:

```text
outputs/tables/final_model_comparison.csv
outputs/tables/final_robustness_comparison.csv
outputs/tables/final_per_class_comparison.csv
outputs/tables/error_examples_sentiment.csv
outputs/tables/error_examples_topic.csv
outputs/tables/error_transition_summary_sentiment.csv
outputs/tables/error_transition_summary_topic.csv
outputs/figures/final_clean_macro_f1_comparison.png
outputs/figures/final_robustness_macro_f1_sentiment.png
outputs/figures/final_robustness_macro_f1_topic.png
outputs/figures/final_robustness_drop_sentiment.png
outputs/figures/final_robustness_drop_topic.png
outputs/reports/final_comparison_report.md
```

Sau Stage 7, dự án đã có đủ kết quả để viết báo cáo chính.

Giai đoạn tiếp theo có hai lựa chọn:

```text
Option A:
Làm Stage 8 — Text Normalization / Robustness Improvement nếu còn thời gian.

Option B:
Bỏ Stage 8 khỏi thực nghiệm chính, đưa vào hướng phát triển và chuyển sang viết báo cáo.
```